# 提交因子:`f001_realized_skew`

> ⚠️ 平台只执行下方 code cell 里的 `main()`;本 markdown 仅供人阅读,平台忽略。
> **定义函数不产生任何输出 —— 直接上传即可,无需运行。**

---

```
f001_realized_skew —— 日内已实现偏度(20 日平滑)

【因子定义】
  r(t)    = ln(close_t / close_{t-1})        # 仅日内相邻分钟;跨日首根因 LAG 为空被剔除 → 无隔夜污染
  S(i,d)  = skewness({ r(t) : t∈日d })        # DuckDB 内置 skewness();当日要求 ≥30 个有效收益且方差>0
  S̄(i,d) = mean(S over 过去 20 交易日)        # 滚动平滑去噪;窗口内要求 ≥10 个有效日
  factor  = -1 × S̄(i,d)                       # 偏度与未来收益负相关 → 取负使"因子值越大越好"

【经济假设(答辩用)】
  散户的"彩票偏好":右偏(有暴涨尾巴)的股票被买贵 → 未来收益偏低;左偏股票被嫌弃 → 未来收益偏高。
  中证 1000 小盘股散户占比高,该效应最强。

【预期正交维度(BARRA 覆盖不到)】
  偏度是收益分布的"三阶矩",而 BARRA 风格(市值/价值/动量/波动率)全是一/二阶矩。
  三阶矩信号经平台"BARRA 风格剔除(取残差)"后几乎不衰减,且与公开因子库(多为价量/价值类)大概率低相关
  → Elastic Net 易选中、权重稳(利好 B 项 ModelScore)。

【AI 介入点】
  无。本因子是人工设计的 **基线 / 打分管线测试因子**(纯人工构建,非 AI)。
  AI 子赛道的正式提交必须由 LLM/RL/GA/NN 主导构建,链路另置于 ../ai_gen/;
  严禁给本因子套个 AI 外壳充数(CLAUDE.md 红线:伪 AI 直接判负 L4)。

【无 look-ahead】
  d 日因子仅用 d-19..d 已收盘的分钟数据预测 d+1;平滑窗只含过去 + 当日,不含任何未来信息。

参考:Amaya, Christoffersen, Jacobs, Vasquez (2015), "Does Realized Skewness Predict
the Cross-Section of Equity Returns?", JFE;开源证券《高频偏度因子》系列。
```


In [ ]:
def main(datasource, start_date, end_date):
    """计算日内已实现偏度(20 日平滑)因子。

    Args:
        datasource (str): 数据表名,如 'bigalpha_factor_2026_stock_bar1m'。
        start_date (str): 起始日 'YYYY-MM-DD HH:MM:SS'。
        end_date (str): 截止日 'YYYY-MM-DD HH:MM:SS'。

    Returns:
        pd.DataFrame: 列严格为 ['date', 'instrument', 'factor'],日频,因子值越大越好。
    """
    import pandas as pd
    import dai

    # 全部计算压进 DAI(DuckDB 方言)的一条 SQL,避免把上亿行 1 分钟数据拉回 pandas。
    sql = f"""
        WITH ret AS (                          -- 日内相邻分钟对数收益(跨日首根 LAG 为空 → 被剔除)
            SELECT
                instrument,
                date::DATE AS d,
                ln(close / NULLIF(LAG(close) OVER (
                    PARTITION BY instrument, date::DATE ORDER BY date), 0)) AS r
            FROM {datasource}
        ),
        daily AS (                             -- 每股每日偏度;卡 ≥30 有效收益且方差>0(剔除一字板/极不活跃)
            SELECT instrument, d, skewness(r) AS s
            FROM ret
            WHERE r IS NOT NULL
            GROUP BY instrument, d
            HAVING count(r) >= 30 AND stddev_samp(r) > 0
        ),
        smooth AS (                            -- 滚动 20 交易日平滑(只用过去 + 当日,无未来)
            SELECT
                instrument, d,
                avg(s)   OVER w AS s_bar,
                count(s) OVER w AS nd
            FROM daily
            WINDOW w AS (
                PARTITION BY instrument ORDER BY d
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
        )
        SELECT
            d::DATETIME AS date,
            instrument,
            -1.0 * s_bar AS factor             -- 偏度负相关 → 取负;"越大越好"
        FROM smooth
        WHERE nd >= 10                         -- 滚动窗内 ≥10 个有效日才出值(否则平滑不可靠 → 留空)
    """

    # 回看缓冲:20 交易日平滑需要 start_date 之前约 40 个日历日(覆盖含春节的假期)的原始分钟数据,
    # 保证请求区间的"第一天"就已有完整的平滑值。
    lookback_days = 40
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=lookback_days)

    df = dai.query(sql, filters={'date': [q_start, end_date]}, compression=True).df()

    # 裁回精确请求区间(缓冲期数据只用于喂平滑窗,不作为输出)。
    df = df[df['date'].between(start_date, end_date)]
    return df[['date', 'instrument', 'factor']]